AE + GSA + RF for feature selection 

IMPORTANT NOTE:

**USING CV = 5**
**CLASS_WEIGHT = 'BALANCED'**



In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold  
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
import warnings

warnings.filterwarnings('ignore')
# Uses GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Data Loading & Preparation
df = pd.read_excel('../minmax.xlsx')
data = df.values
labels = pd.read_csv('../idC_with_header.csv').values.flatten() - 1  # Adjust labels to be in range [0, 13]

X_tensor = torch.tensor(data, dtype=torch.float32)
y_tensor = torch.tensor(labels, dtype=torch.long)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_tensor, y_tensor, test_size=0.2, random_state=42
)

batch_size = 32
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 2. Define the Joint Autoencoder-Classifier Model
class JointAutoencoderClassifier(nn.Module):
    def __init__(self, input_dim, latent_dim=32, num_classes=14):
        super(JointAutoencoderClassifier, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, input_dim),
            nn.Tanh()  # Ensure your input is in [-1, 1]. Use Sigmoid() if [0, 1]
        )
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )
    
    def forward(self, x):
        latent = self.encoder(x)
        reconstruction = self.decoder(latent)
        logits = self.classifier(latent)
        return reconstruction, logits

# 3. Loss Functions & Optimizer
reconstruction_loss_fn = nn.MSELoss()
classification_loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
lambda_recon = 0.5

def combined_loss(reconstructed, original, logits, labels):
    loss_recon = reconstruction_loss_fn(reconstructed, original)
    loss_class = classification_loss_fn(logits, labels)
    return lambda_recon * loss_recon + (1 - lambda_recon) * loss_class

input_dim = data.shape[1]
num_classes = 14
latent_dim = 32

model = JointAutoencoderClassifier(input_dim, latent_dim, num_classes).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
num_epochs = 50

# 4. Joint Training
print("=== Joint Training Stage ===")
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        reconstruction, logits = model(inputs)
        loss = combined_loss(reconstruction, inputs, logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss/len(train_loader):.4f}")

# 5. Extract Latent Features
model.eval()
with torch.no_grad():
    train_latent = model.encoder(X_train.to(device)).cpu().numpy()
    test_latent = model.encoder(X_test.to(device)).cpu().numpy()

y_train_np = y_train.numpy()
y_test_np = y_test.numpy()



# 6. Modified GSA Feature Selection
def gsa_optimize(features, labels, num_agents=10, max_iterations=20):
    num_features = features.shape[1]
    G0 = 100  # Initial gravitational constant
    epsilon = 1e-6

    # Initialize binary positions and velocities
    positions = np.random.randint(0, 2, size=(num_agents, num_features))
    velocities = np.zeros_like(positions, dtype=float)

    # Ensure at least one feature selected per agent
    for pos in positions:
        if not pos.any():
            pos[np.random.randint(0, num_features)] = 1

    def fitness_function(position):
        selected = np.where(position == 1)[0]
        if len(selected) == 0:
            return -1
        try:
            model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
            scores = cross_val_score(model, features[:, selected], labels, cv=StratifiedKFold(n_splits=5), scoring='accuracy')
            return np.mean(scores)
        except:
            return -1

    for iteration in range(max_iterations):
        fitness = np.array([fitness_function(agent) for agent in positions])

        # Normalize fitness to masses (higher fitness = more mass)
        best, worst = fitness.max(), fitness.min()
        if best == worst:
            mass = np.ones(num_agents)
        else:
            mass = (fitness - worst) / (best - worst + epsilon)
        mass = mass / mass.sum()  # Normalize

        # Update gravitational constant
        G = G0 * np.exp(-20 * iteration / max_iterations)

        # Calculate total force on each agent
        for i in range(num_agents):
            force = np.zeros(num_features)
            for j in range(num_agents):
                if i != j:
                    distance = np.linalg.norm(positions[i] - positions[j]) + epsilon
                    rand_coeff = np.random.rand(num_features)
                    force += rand_coeff * G * (mass[i] * mass[j]) / distance * (positions[j] - positions[i])
            # Acceleration: F = m * a => a = F / m
            acceleration = force / (mass[i] + epsilon)
            # Velocity update
            velocities[i] = np.random.rand(num_features) * velocities[i] + acceleration
            # Position update using sigmoid
            sigmoid = 1 / (1 + np.exp(-velocities[i]))
            positions[i] = np.where(np.random.rand(num_features) < sigmoid, 1, 0)

            # Ensure at least one feature is selected
            if not positions[i].any():
                positions[i][np.random.randint(0, num_features)] = 1

        best_idx = np.argmax(fitness)
        print(f"Iteration {iteration + 1}: Best Fitness = {fitness[best_idx]:.4f}")
        print(f"Selected Features: {np.where(positions[best_idx] == 1)[0]}")

    final_best_idx = np.argmax(fitness)
    return np.where(positions[final_best_idx] == 1)[0]

selected_features = gsa_optimize(train_latent, y_train_np)

# 7. Train & Evaluate SVM
if len(selected_features) > 0:
    print(f"\nFinal selected features: {selected_features}")
    
    # # SVM Classifier
    
    # svm_model = SVC(random_state=42, class_weight='balanced')
    # svm_model.fit(train_latent[:, selected_features], y_train_np)
    # svm_preds = svm_model.predict(test_latent[:, selected_features])
    # svm_acc = accuracy_score(y_test_np, svm_preds)
    # print('\nSVM Performance with BGWO-selected features:')
    # print('Confusion Matrix:\n', confusion_matrix(y_test_np, svm_preds))
    # print('\nClassification Report:\n', classification_report(y_test_np, svm_preds))
    # print(f"\nSVM Accuracy: {svm_acc * 100:.2f}%")
    
    #Random Forest Classifier
    
    # rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
    # rf_model.fit(train_latent[:, selected_features], y_train_np)
    # rf_preds = rf_model.predict(test_latent[:, selected_features])
    # rf_acc = accuracy_score(y_test_np, rf_preds)
    
    # print('\nRandom Forest Performance with BGWO-selected features:')
    # print('Confusion Matrix:\n', confusion_matrix(y_test_np, rf_preds))
    # print('\nClassification Report:\n', classification_report(y_test_np, rf_preds))
    # print(f"\nRF Accuracy: {rf_acc * 100:.2f}%")
    
    
    # Naive Bayes Classifier
    
    nb_model = GaussianNB()
    nb_model.fit(train_latent[:, selected_features], y_train_np)
    nb_preds = nb_model.predict(test_latent[:, selected_features])
    nb_acc = accuracy_score(y_test_np, nb_preds)
    
    print('\nNaive Bayes Performance with BGWO-selected features:')
    print('Confusion Matrix:\n', confusion_matrix(y_test_np, nb_preds))
    print('\nClassification Report:\n', classification_report(y_test_np, nb_preds))
    print(f"\nNBC Accuracy: {nb_acc * 100:.2f}%")
    
    
     
else:
    # Fallback: Use all features if BGWO fails
    print("\nWarning: BGWO selected no features. Using all features as fallback.")
    
    # SVM Classifier
    
    # svm_model = SVC(random_state=42, class_weight='balanced')
    # svm_model.fit(train_latent, y_train_np)
    # svm_preds = svm_model.predict(test_latent)
    # svm_acc = accuracy_score(y_test_np, svm_preds)
    
    # print('\nSVM Performance (using all features):')
    # print('Confusion Matrix:\n', confusion_matrix(y_test_np, svm_preds))
    # print('\nClassification Report:\n', classification_report(y_test_np, svm_preds))
    # print(f"\nSVM Accuracy: {svm_acc * 100:.2f}%")
    
    
    #Random Forest Classifier
    
    # rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
    # rf_model.fit(train_latent[:, selected_features], y_train_np)
    # rf_preds = rf_model.predict(test_latent[:, selected_features])
    # rf_acc = accuracy_score(y_test_np, rf_preds)
    
    # print('\nRandom Forest Performance with BGWO-selected features:')
    # print('Confusion Matrix:\n', confusion_matrix(y_test_np, rf_preds))
    # print('\nClassification Report:\n', classification_report(y_test_np, rf_preds))
    # print(f"\nRF Accuracy: {rf_acc * 100:.2f}%")
    
    # Naive Bayes Classifier 
    
    nb_model = GaussianNB()
    nb_model.fit(train_latent, y_train_np)
    nb_preds = nb_model.predict(test_latent)
    nb_acc = accuracy_score(y_test_np, nb_preds)
    
    print('\nNaive Bayes Performance (using all features):')
    print('Confusion Matrix:\n', confusion_matrix(y_test_np, nb_preds))
    print('\nClassification Report:\n', classification_report(y_test_np, nb_preds))
    print(f"\nNBC Accuracy: {nb_acc * 100:.2f}%")

=== Joint Training Stage ===
Epoch [1/50], Loss: 1.1488
Epoch [2/50], Loss: 0.7804
Epoch [3/50], Loss: 0.5911
Epoch [4/50], Loss: 0.5323
Epoch [5/50], Loss: 0.5217
Epoch [6/50], Loss: 0.4556
Epoch [7/50], Loss: 0.4138
Epoch [8/50], Loss: 0.4903
Epoch [9/50], Loss: 0.4229
Epoch [10/50], Loss: 0.4100
Epoch [11/50], Loss: 0.4074
Epoch [12/50], Loss: 0.3859
Epoch [13/50], Loss: 0.4043
Epoch [14/50], Loss: 0.4045
Epoch [15/50], Loss: 0.4042
Epoch [16/50], Loss: 0.4070
Epoch [17/50], Loss: 0.4026
Epoch [18/50], Loss: 0.4653
Epoch [19/50], Loss: 0.4021
Epoch [20/50], Loss: 0.3649
Epoch [21/50], Loss: 0.3995
Epoch [22/50], Loss: 0.3804
Epoch [23/50], Loss: 0.3819
Epoch [24/50], Loss: 0.4378
Epoch [25/50], Loss: 0.3922
Epoch [26/50], Loss: 0.3929
Epoch [27/50], Loss: 0.3855
Epoch [28/50], Loss: 0.3975
Epoch [29/50], Loss: 0.3650
Epoch [30/50], Loss: 0.3692
Epoch [31/50], Loss: 0.3802
Epoch [32/50], Loss: 0.3692
Epoch [33/50], Loss: 0.3972
Epoch [34/50], Loss: 0.3825
Epoch [35/50], Loss: 0.4201


c:\Users\a2004\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\a2004\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\a2004\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\a2004\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\a2004\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_split.

Iteration 1: Best Fitness = 1.0000
Selected Features: [ 7  8  9 10 11 13 14 16 18 19 21 24 25 27 28 29 30 31]


c:\Users\a2004\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\a2004\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\a2004\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\a2004\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\a2004\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_split.

Iteration 2: Best Fitness = 1.0000
Selected Features: [ 0  1  8 10 11 12 13 15 16 19 23 26 27 30 31]


...

Iteration 18: Best Fitness = 1.0000
Selected Features: [ 0  3 11 15 18 24 27 28]


c:\Users\a2004\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\a2004\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\a2004\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\a2004\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\a2004\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_split.

Iteration 19: Best Fitness = 1.0000
Selected Features: [ 4  5  6  8  9 10 11 12 13 15 18 20 23 25 30 31]


c:\Users\a2004\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\a2004\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\a2004\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\a2004\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\a2004\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_split.

Iteration 20: Best Fitness = 1.0000
Selected Features: [ 1  2  3  4  5  8 11 12 14 16 17 18 22 23 24 28 30 31]

Final selected features: [ 1  2  3  4  5  8 11 12 14 16 17 18 22 23 24 28 30 31]

Naive Bayes Performance with BGWO-selected features:
Confusion Matrix:
 [[ 2  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  8  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  6  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  2 16  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  9  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  1  5  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  1  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0 13  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  1  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  5  0  0  0]
 [ 0  0  0  0  0  1  0  0  0  0  5  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  7  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  7]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00         2
           1       1.00      1.00      1.00         8

AE + GSA + SVM for feature selection 

IMPORTANT NOTE:

**USING CV = 5**
**CLASS_WEIGHT = 'BALANCED'**


In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold  
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
import warnings

warnings.filterwarnings('ignore')

# Uses GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Data Loading & Preparation
df = pd.read_excel('../minmax.xlsx')
data = df.values
labels = pd.read_csv('../idC_with_header.csv').values.flatten() - 1  # Adjust labels to be in range [0, 13]

X_tensor = torch.tensor(data, dtype=torch.float32)
y_tensor = torch.tensor(labels, dtype=torch.long)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_tensor, y_tensor, test_size=0.2, random_state=42
)

batch_size = 32
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 2. Define the Joint Autoencoder-Classifier Model
class JointAutoencoderClassifier(nn.Module):
    def __init__(self, input_dim, latent_dim=32, num_classes=14):
        super(JointAutoencoderClassifier, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, input_dim),
            nn.Tanh()  # Ensure your input is in [-1, 1]. Use Sigmoid() if [0, 1]
        )
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )
    
    def forward(self, x):
        latent = self.encoder(x)
        reconstruction = self.decoder(latent)
        logits = self.classifier(latent)
        return reconstruction, logits

# 3. Loss Functions & Optimizer
reconstruction_loss_fn = nn.MSELoss()
classification_loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
lambda_recon = 0.5

def combined_loss(reconstructed, original, logits, labels):
    loss_recon = reconstruction_loss_fn(reconstructed, original)
    loss_class = classification_loss_fn(logits, labels)
    return lambda_recon * loss_recon + (1 - lambda_recon) * loss_class

input_dim = data.shape[1]
num_classes = 14
latent_dim = 32

model = JointAutoencoderClassifier(input_dim, latent_dim, num_classes).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
num_epochs = 50

# 4. Joint Training
print("=== Joint Training Stage ===")
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        reconstruction, logits = model(inputs)
        loss = combined_loss(reconstruction, inputs, logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss/len(train_loader):.4f}")

# 5. Extract Latent Features
model.eval()
with torch.no_grad():
    train_latent = model.encoder(X_train.to(device)).cpu().numpy()
    test_latent = model.encoder(X_test.to(device)).cpu().numpy()

y_train_np = y_train.numpy()
y_test_np = y_test.numpy()



# 6. Modified GSA Feature Selection
def gsa_optimize(features, labels, num_agents=10, max_iterations=20):
    num_features = features.shape[1]
    G0 = 100  # Initial gravitational constant
    epsilon = 1e-6

    # Initialize binary positions and velocities
    positions = np.random.randint(0, 2, size=(num_agents, num_features))
    velocities = np.zeros_like(positions, dtype=float)

    # Ensure at least one feature selected per agent
    for pos in positions:
        if not pos.any():
            pos[np.random.randint(0, num_features)] = 1

    def fitness_function(position):
        selected = np.where(position == 1)[0]
        if len(selected) == 0:
            return -1
        try:
            model = SVC(random_state=42, class_weight='balanced')
            scores = cross_val_score(model, features[:, selected], labels, cv=StratifiedKFold(n_splits=5), scoring='accuracy')
            return np.mean(scores)
        except:
            return -1

    for iteration in range(max_iterations):
        fitness = np.array([fitness_function(agent) for agent in positions])

        # Normalize fitness to masses (higher fitness = more mass)
        best, worst = fitness.max(), fitness.min()
        if best == worst:
            mass = np.ones(num_agents)
        else:
            mass = (fitness - worst) / (best - worst + epsilon)
        mass = mass / mass.sum()  # Normalize

        # Update gravitational constant
        G = G0 * np.exp(-20 * iteration / max_iterations)

        # Calculate total force on each agent
        for i in range(num_agents):
            force = np.zeros(num_features)
            for j in range(num_agents):
                if i != j:
                    distance = np.linalg.norm(positions[i] - positions[j]) + epsilon
                    rand_coeff = np.random.rand(num_features)
                    force += rand_coeff * G * (mass[i] * mass[j]) / distance * (positions[j] - positions[i])
            # Acceleration: F = m * a => a = F / m
            acceleration = force / (mass[i] + epsilon)
            # Velocity update
            velocities[i] = np.random.rand(num_features) * velocities[i] + acceleration
            # Position update using sigmoid
            sigmoid = 1 / (1 + np.exp(-velocities[i]))
            positions[i] = np.where(np.random.rand(num_features) < sigmoid, 1, 0)

            # Ensure at least one feature is selected
            if not positions[i].any():
                positions[i][np.random.randint(0, num_features)] = 1

        best_idx = np.argmax(fitness)
        print(f"Iteration {iteration + 1}: Best Fitness = {fitness[best_idx]:.4f}")
        print(f"Selected Features: {np.where(positions[best_idx] == 1)[0]}")

    final_best_idx = np.argmax(fitness)
    return np.where(positions[final_best_idx] == 1)[0]

selected_features = gsa_optimize(train_latent, y_train_np)

# 7. Train & Evaluate SVM
if len(selected_features) > 0:
    print(f"\nFinal selected features: {selected_features}")
    
    # SVM Classifier
    # svm_model = SVC(random_state=42, class_weight='balanced')
    # svm_model.fit(train_latent[:, selected_features], y_train_np)
    # svm_preds = svm_model.predict(test_latent[:, selected_features])
    # svm_acc = accuracy_score(y_test_np, svm_preds)
    # print('\nSVM Performance with BGWO-selected features:')
    # print('Confusion Matrix:\n', confusion_matrix(y_test_np, svm_preds))
    # print('\nClassification Report:\n', classification_report(y_test_np, svm_preds))
    # print(f"\nSVM Accuracy: {svm_acc * 100:.2f}%")
    
    # Random Forest Classifier
    
    # rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
    # rf_model.fit(train_latent[:, selected_features], y_train_np)
    # rf_preds = rf_model.predict(test_latent[:, selected_features])
    # rf_acc = accuracy_score(y_test_np, rf_preds)
    
    # print('\nRandom Forest Performance with BGWO-selected features:')
    # print('Confusion Matrix:\n', confusion_matrix(y_test_np, rf_preds))
    # print('\nClassification Report:\n', classification_report(y_test_np, rf_preds))
    # print(f"\nRF Accuracy: {rf_acc * 100:.2f}%")
    
    
    # Naive Bayes Classifier
    
    nb_model = GaussianNB()
    nb_model.fit(train_latent[:, selected_features], y_train_np)
    nb_preds = nb_model.predict(test_latent[:, selected_features])
    nb_acc = accuracy_score(y_test_np, nb_preds)
    
    print('\nNaive Bayes Performance with BGWO-selected features:')
    print('Confusion Matrix:\n', confusion_matrix(y_test_np, nb_preds))
    print('\nClassification Report:\n', classification_report(y_test_np, nb_preds))
    print(f"\nNBC Accuracy: {nb_acc * 100:.2f}%")
    
    
     
else:
    # Fallback: Use all features if BGWO fails
    print("\nWarning: BGWO selected no features. Using all features as fallback.")
    
    # SVM Classifier
    
    # svm_model = SVC(random_state=42, class_weight='balanced')
    # svm_model.fit(train_latent, y_train_np)
    # svm_preds = svm_model.predict(test_latent)
    # svm_acc = accuracy_score(y_test_np, svm_preds)
    
    # print('\nSVM Performance (using all features):')
    # print('Confusion Matrix:\n', confusion_matrix(y_test_np, svm_preds))
    # print('\nClassification Report:\n', classification_report(y_test_np, svm_preds))
    # print(f"\nSVM Accuracy: {svm_acc * 100:.2f}%")
    
    
    # Random Forest Classifier
    
    # rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
    # rf_model.fit(train_latent[:, selected_features], y_train_np)
    # rf_preds = rf_model.predict(test_latent[:, selected_features])
    # rf_acc = accuracy_score(y_test_np, rf_preds)
    
    # print('\nRandom Forest Performance with BGWO-selected features:')
    # print('Confusion Matrix:\n', confusion_matrix(y_test_np, rf_preds))
    # print('\nClassification Report:\n', classification_report(y_test_np, rf_preds))
    # print(f"\nRF Accuracy: {rf_acc * 100:.2f}%")
    
    # Naive Bayes Classifier 
    nb_model = GaussianNB()
    nb_model.fit(train_latent, y_train_np)
    nb_preds = nb_model.predict(test_latent)
    nb_acc = accuracy_score(y_test_np, nb_preds)
    
    print('\nNaive Bayes Performance (using all features):')
    print('Confusion Matrix:\n', confusion_matrix(y_test_np, nb_preds))
    print('\nClassification Report:\n', classification_report(y_test_np, nb_preds))
    print(f"\nNBC Accuracy: {nb_acc * 100:.2f}%")

=== Joint Training Stage ===
Epoch [1/50], Loss: 1.1803
Epoch [2/50], Loss: 0.7528
Epoch [3/50], Loss: 0.6181
Epoch [4/50], Loss: 0.5893
Epoch [5/50], Loss: 0.4899
Epoch [6/50], Loss: 0.4891
Epoch [7/50], Loss: 0.4517
Epoch [8/50], Loss: 0.4778
Epoch [9/50], Loss: 0.4535
Epoch [10/50], Loss: 0.4642
Epoch [11/50], Loss: 0.4274
Epoch [12/50], Loss: 0.3805
Epoch [13/50], Loss: 0.4300
Epoch [14/50], Loss: 0.3683
Epoch [15/50], Loss: 0.4177
Epoch [16/50], Loss: 0.3853
Epoch [17/50], Loss: 0.3931
Epoch [18/50], Loss: 0.4211
Epoch [19/50], Loss: 0.3777
Epoch [20/50], Loss: 0.3535
Epoch [21/50], Loss: 0.4158
Epoch [22/50], Loss: 0.4115
Epoch [23/50], Loss: 0.4193
Epoch [24/50], Loss: 0.3744
Epoch [25/50], Loss: 0.3547
Epoch [26/50], Loss: 0.3780
Epoch [27/50], Loss: 0.4294
Epoch [28/50], Loss: 0.4218
Epoch [29/50], Loss: 0.3732
Epoch [30/50], Loss: 0.4086
Epoch [31/50], Loss: 0.3855
Epoch [32/50], Loss: 0.3668
Epoch [33/50], Loss: 0.4375
Epoch [34/50], Loss: 0.3373
Epoch [35/50], Loss: 0.4206


AE + GSA + NB for feature selection 

IMPORTANT NOTE:

**USING CV = 5**
**CLASS_WEIGHT = 'BALANCED'**

FOR EXAMPLE OUTPUTS WARNINGS BECAUSE A CLASS HAS 2 MEMBERS ONLY IN THE DATA SET, AS A RESULT I CHANGED THE CV TO STRIFIEDKFOLD N_SPLITS = 5, AND ADDED A PARAMETER CLASS_WEIGHT = 'BALANCED' ADDRESS THIS ISSUE. the performance metrics went from 0.93 to 0.99 and accuracy from 0.93 to 0.98

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold  
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
import warnings

warnings.filterwarnings('ignore')

# Uses GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Data Loading & Preparation
df = pd.read_excel('../minmax.xlsx')
data = df.values
labels = pd.read_csv('../idC_with_header.csv').values.flatten() - 1  # Adjust labels to be in range [0, 13]

X_tensor = torch.tensor(data, dtype=torch.float32)
y_tensor = torch.tensor(labels, dtype=torch.long)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_tensor, y_tensor, test_size=0.2, random_state=42
)

batch_size = 32
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 2. Define the Joint Autoencoder-Classifier Model
class JointAutoencoderClassifier(nn.Module):
    def __init__(self, input_dim, latent_dim=32, num_classes=14):
        super(JointAutoencoderClassifier, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, input_dim),
            nn.Tanh()  # Ensure your input is in [-1, 1]. Use Sigmoid() if [0, 1]
        )
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )
    
    def forward(self, x):
        latent = self.encoder(x)
        reconstruction = self.decoder(latent)
        logits = self.classifier(latent)
        return reconstruction, logits

# 3. Loss Functions & Optimizer
reconstruction_loss_fn = nn.MSELoss()
classification_loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
lambda_recon = 0.5

def combined_loss(reconstructed, original, logits, labels):
    loss_recon = reconstruction_loss_fn(reconstructed, original)
    loss_class = classification_loss_fn(logits, labels)
    return lambda_recon * loss_recon + (1 - lambda_recon) * loss_class

input_dim = data.shape[1]
num_classes = 14
latent_dim = 32

model = JointAutoencoderClassifier(input_dim, latent_dim, num_classes).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
num_epochs = 50

# 4. Joint Training
print("=== Joint Training Stage ===")
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        reconstruction, logits = model(inputs)
        loss = combined_loss(reconstruction, inputs, logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss/len(train_loader):.4f}")

# 5. Extract Latent Features
model.eval()
with torch.no_grad():
    train_latent = model.encoder(X_train.to(device)).cpu().numpy()
    test_latent = model.encoder(X_test.to(device)).cpu().numpy()

y_train_np = y_train.numpy()
y_test_np = y_test.numpy()



# 6. Modified GSA Feature Selection
def gsa_optimize(features, labels, num_agents=10, max_iterations=20):
    num_features = features.shape[1]
    G0 = 100  # Initial gravitational constant
    epsilon = 1e-6

    # Initialize binary positions and velocities
    positions = np.random.randint(0, 2, size=(num_agents, num_features))
    velocities = np.zeros_like(positions, dtype=float)

    # Ensure at least one feature selected per agent
    for pos in positions:
        if not pos.any():
            pos[np.random.randint(0, num_features)] = 1

    def fitness_function(position):
        selected = np.where(position == 1)[0]
        if len(selected) == 0:
            return -1
        try:
            model = GaussianNB()
            scores = cross_val_score(model, features[:, selected], labels, cv=StratifiedKFold(n_splits=5), scoring='accuracy')
            return np.mean(scores)
        except:
            return -1

    for iteration in range(max_iterations):
        fitness = np.array([fitness_function(agent) for agent in positions])

        # Normalize fitness to masses (higher fitness = more mass)
        best, worst = fitness.max(), fitness.min()
        if best == worst:
            mass = np.ones(num_agents)
        else:
            mass = (fitness - worst) / (best - worst + epsilon)
        mass = mass / mass.sum()  # Normalize

        # Update gravitational constant
        G = G0 * np.exp(-20 * iteration / max_iterations)

        # Calculate total force on each agent
        for i in range(num_agents):
            force = np.zeros(num_features)
            for j in range(num_agents):
                if i != j:
                    distance = np.linalg.norm(positions[i] - positions[j]) + epsilon
                    rand_coeff = np.random.rand(num_features)
                    force += rand_coeff * G * (mass[i] * mass[j]) / distance * (positions[j] - positions[i])
            # Acceleration: F = m * a => a = F / m
            acceleration = force / (mass[i] + epsilon)
            # Velocity update
            velocities[i] = np.random.rand(num_features) * velocities[i] + acceleration
            # Position update using sigmoid
            sigmoid = 1 / (1 + np.exp(-velocities[i]))
            positions[i] = np.where(np.random.rand(num_features) < sigmoid, 1, 0)

            # Ensure at least one feature is selected
            if not positions[i].any():
                positions[i][np.random.randint(0, num_features)] = 1

        best_idx = np.argmax(fitness)
        print(f"Iteration {iteration + 1}: Best Fitness = {fitness[best_idx]:.4f}")
        print(f"Selected Features: {np.where(positions[best_idx] == 1)[0]}")

    final_best_idx = np.argmax(fitness)
    return np.where(positions[final_best_idx] == 1)[0]

selected_features = gsa_optimize(train_latent, y_train_np)

# 7. Train & Evaluate SVM
if len(selected_features) > 0:
    print(f"\nFinal selected features: {selected_features}")
    
    # SVM Classifier
    svm_model = SVC(random_state=42, class_weight='balanced')
    svm_model.fit(train_latent[:, selected_features], y_train_np)
    svm_preds = svm_model.predict(test_latent[:, selected_features])
    svm_acc = accuracy_score(y_test_np, svm_preds)
    print('\nSVM Performance with BGWO-selected features:')
    print('Confusion Matrix:\n', confusion_matrix(y_test_np, svm_preds))
    print('\nClassification Report:\n', classification_report(y_test_np, svm_preds))
    print(f"\nSVM Accuracy: {svm_acc * 100:.2f}%")
    
    # Random Forest Classifier
    
    # rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
    # rf_model.fit(train_latent[:, selected_features], y_train_np)
    # rf_preds = rf_model.predict(test_latent[:, selected_features])
    # rf_acc = accuracy_score(y_test_np, rf_preds)
    
    # print('\nRandom Forest Performance with BGWO-selected features:')
    # print('Confusion Matrix:\n', confusion_matrix(y_test_np, rf_preds))
    # print('\nClassification Report:\n', classification_report(y_test_np, rf_preds))
    # print(f"\nRF Accuracy: {rf_acc * 100:.2f}%")
    
    
    # Naive Bayes Classifier
    
    # nb_model = GaussianNB()
    # nb_model.fit(train_latent[:, selected_features], y_train_np)
    # nb_preds = nb_model.predict(test_latent[:, selected_features])
    # nb_acc = accuracy_score(y_test_np, nb_preds)
    
    # print('\nNaive Bayes Performance with BGWO-selected features:')
    # print('Confusion Matrix:\n', confusion_matrix(y_test_np, nb_preds))
    # print('\nClassification Report:\n', classification_report(y_test_np, nb_preds))
    # print(f"\nNBC Accuracy: {nb_acc * 100:.2f}%")
    
    
     
else:
    # Fallback: Use all features if BGWO fails
    print("\nWarning: BGWO selected no features. Using all features as fallback.")
    
    # SVM Classifier
    
    svm_model = SVC(random_state=42, class_weight='balanced')
    svm_model.fit(train_latent, y_train_np)
    svm_preds = svm_model.predict(test_latent)
    svm_acc = accuracy_score(y_test_np, svm_preds)
    
    print('\nSVM Performance (using all features):')
    print('Confusion Matrix:\n', confusion_matrix(y_test_np, svm_preds))
    print('\nClassification Report:\n', classification_report(y_test_np, svm_preds))
    print(f"\nSVM Accuracy: {svm_acc * 100:.2f}%")
    
    
    # Random Forest Classifier
    
    # rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
    # rf_model.fit(train_latent[:, selected_features], y_train_np)
    # rf_preds = rf_model.predict(test_latent[:, selected_features])
    # rf_acc = accuracy_score(y_test_np, rf_preds)
    
    # print('\nRandom Forest Performance with BGWO-selected features:')
    # print('Confusion Matrix:\n', confusion_matrix(y_test_np, rf_preds))
    # print('\nClassification Report:\n', classification_report(y_test_np, rf_preds))
    # print(f"\nRF Accuracy: {rf_acc * 100:.2f}%")
    
    # Naive Bayes Classifier 
    # nb_model = GaussianNB()
    # nb_model.fit(train_latent, y_train_np)
    # nb_preds = nb_model.predict(test_latent)
    # nb_acc = accuracy_score(y_test_np, nb_preds)
    
    # print('\nNaive Bayes Performance (using all features):')
    # print('Confusion Matrix:\n', confusion_matrix(y_test_np, nb_preds))
    # print('\nClassification Report:\n', classification_report(y_test_np, nb_preds))
    # print(f"\nNBC Accuracy: {nb_acc * 100:.2f}%")

=== Joint Training Stage ===
Epoch [1/50], Loss: 1.2147
Epoch [2/50], Loss: 0.8297
Epoch [3/50], Loss: 0.7097
Epoch [4/50], Loss: 0.6068
Epoch [5/50], Loss: 0.5547
Epoch [6/50], Loss: 0.4677
Epoch [7/50], Loss: 0.5080
Epoch [8/50], Loss: 0.4891
Epoch [9/50], Loss: 0.4627
Epoch [10/50], Loss: 0.3912
Epoch [11/50], Loss: 0.4104
Epoch [12/50], Loss: 0.4062
Epoch [13/50], Loss: 0.4143
Epoch [14/50], Loss: 0.4141
Epoch [15/50], Loss: 0.3820
Epoch [16/50], Loss: 0.4377
Epoch [17/50], Loss: 0.4080
Epoch [18/50], Loss: 0.3792
Epoch [19/50], Loss: 0.4527
Epoch [20/50], Loss: 0.4149
Epoch [21/50], Loss: 0.3963
Epoch [22/50], Loss: 0.3933
Epoch [23/50], Loss: 0.3947
Epoch [24/50], Loss: 0.3812
Epoch [25/50], Loss: 0.3810
Epoch [26/50], Loss: 0.3943
Epoch [27/50], Loss: 0.3769
Epoch [28/50], Loss: 0.4012
Epoch [29/50], Loss: 0.3744
Epoch [30/50], Loss: 0.4241
Epoch [31/50], Loss: 0.3750
Epoch [32/50], Loss: 0.3498
Epoch [33/50], Loss: 0.3903
Epoch [34/50], Loss: 0.4413
Epoch [35/50], Loss: 0.3760
